# Chapter 7: Inference for Numerical Data
**Haydar Kilic — Artificial Intelligence Engineering**

This notebook covers the **t-distribution**, **paired data**, **difference of two means**, and **ANOVA** with hands-on Python examples.

**Topics:**
1. The t-Distribution — What it is, how it differs from Normal
2. One-Sample t-Test — Friday the 13th
3. Paired Data (Paired t-test) — Reading & Writing Scores
4. Two Independent Samples t-Test — Diamond Prices
5. Power Calculation for a Two-Sample Test
6. ANOVA — Wolf River Aldrin Concentrations
7. Multiple Comparisons (Bonferroni Correction)

In [ ]:
# Required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import t, f, norm
import warnings
warnings.filterwarnings('ignore')

# Visual settings
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
sns.set_style('whitegrid')

print('Libraries loaded successfully.')

---
## 1. The t-Distribution

### Why the t-distribution?

When the population standard deviation **σ is unknown** (almost always the case in practice), we estimate the standard error using the sample standard deviation **s**:

$$SE = \frac{s}{\sqrt{n}}$$

However, **when n is small**, s is not a reliable estimate of σ. The **t-distribution** is used to account for this additional uncertainty.

**Properties:**
- **Centred at 0** and symmetric, just like the standard normal
- Has **heavier tails** than the normal distribution
- Single parameter: **degrees of freedom (df = n − 1)**
- As df → ∞, the t-distribution **converges to the normal**

In [ ]:
x          = np.linspace(-5, 5, 500)
normal_pdf = norm.pdf(x)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Normal vs different df values
ax = axes[0]
ax.plot(x, normal_pdf, 'k-', lw=2.5, label='Normal', zorder=5)
colors = ['#e74c3c', '#e67e22', '#3498db', '#95a5a6']
dfs    = [1, 2, 5, 10]
for df_val, color in zip(dfs, colors):
    ax.plot(x, t.pdf(x, df=df_val), color=color, lw=1.8,
            linestyle='--', label=f't, df={df_val}', alpha=0.85)
ax.set_title('t-Distribution vs Normal Distribution')
ax.set_xlabel('Value')
ax.set_ylabel('Density')
ax.legend()
ax.set_xlim(-5, 5)
ax.set_ylim(0, 0.45)

# Right: Heavy-tail emphasis
ax2 = axes[1]
ax2.plot(x, normal_pdf, 'k-', lw=2.5, label='Normal', zorder=5)
ax2.plot(x, t.pdf(x, df=3), '#e74c3c', lw=2, linestyle='--', label='t, df=3')
# Colour the tails
tail_r = x[x > 2]
ax2.fill_between(tail_r, norm.pdf(tail_r), t.pdf(tail_r, df=3),
                 alpha=0.4, color='#e74c3c', label='Extra heavy tail')
tail_l = x[x < -2]
ax2.fill_between(tail_l, norm.pdf(tail_l), t.pdf(tail_l, df=3),
                 alpha=0.4, color='#e74c3c')
ax2.set_title('Heavy Tails of the t-Distribution (df=3)')
ax2.set_xlabel('Value')
ax2.set_ylabel('Density')
ax2.legend()
ax2.set_xlim(-5, 5)
ax2.set_ylim(0, 0.45)

plt.tight_layout()
plt.suptitle('Figure 1: The t-Distribution', fontsize=14, fontweight='bold', y=1.02)
plt.show()

# Numerical comparison
print('=== Normal vs t: P(|X| > 2) probabilities ===')
print(f'Normal:    {2 * norm.sf(2):.4f}')
for df_val in [1, 2, 5, 10, 30]:
    print(f't(df={df_val:2d}): {2 * t.sf(2, df=df_val):.4f}')

**Observation:** As df increases, the t-distribution converges to the normal. Beyond df=30 the difference is negligible.

---
## 2. One-Sample t-Test: Friday the 13th

### Research Question
Traffic flow on the 6th and 13th Fridays in England between 1990 and 1992 was compared. **Do people drive less on Friday the 13th?**

### Hypotheses
$$H_0: \mu_{diff} = 0 \quad \text{(no difference between the 6th and 13th)}$$
$$H_A: \mu_{diff} \neq 0 \quad \text{(two-sided)}$$

### Test Statistic
$$T_{df} = \frac{\bar{x}_{diff} - 0}{SE}, \quad SE = \frac{s_{diff}}{\sqrt{n}}, \quad df = n-1$$

In [ ]:
# Data: traffic flows on the 6th and 13th Fridays
data_friday = pd.DataFrame({
    'date':     ['1990 Jul', '1990 Jul', '1991 Sep', '1991 Sep',
                 '1991 Dec', '1991 Dec', '1992 Mar', '1992 Mar',
                 '1992 Nov', '1992 Nov'],
    'sixth':    [139246, 134012, 137055, 133732, 123552, 121139,
                 128293, 124631, 124609, 117584],
    'thirteenth':[138548, 132908, 136018, 131843, 121641, 118723,
                  125532, 120249, 122770, 117263],
    'location': ['L1','L2','L1','L2','L1','L2','L1','L2','L1','L2']
})

data_friday['diff'] = data_friday['sixth'] - data_friday['thirteenth']

print('=== Friday the 13th Dataset ===')
print(data_friday[['date','location','sixth','thirteenth','diff']].to_string(index=False))

x_bar = data_friday['diff'].mean()
s     = data_friday['diff'].std(ddof=1)
n     = len(data_friday)
se    = s / np.sqrt(n)
df    = n - 1
T_obs = (x_bar - 0) / se
p_val = 2 * t.sf(abs(T_obs), df=df)

print(f'\n=== Summary Statistics ===')
print(f'Mean difference (x̄):  {x_bar:.2f} vehicles')
print(f'Std deviation (s):     {s:.2f}')
print(f'n:                     {n}')
print(f'Standard error (SE):   {se:.2f}')
print(f'\n=== Hypothesis Test ===')
print(f'T statistic:           {T_obs:.3f}')
print(f'Degrees of freedom:    {df}')
print(f'p-value:               {p_val:.6f}')
print(f'\nDecision (α=0.05):     {"REJECT H₀ ✓" if p_val < 0.05 else "FAIL TO REJECT H₀"}')

In [ ]:
# Visualization: difference histogram + t-distribution + p-value
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Histogram of differences
ax1 = axes[0]
ax1.hist(data_friday['diff'], bins=6, color='#3498db', alpha=0.7,
         edgecolor='white', linewidth=0.8)
ax1.axvline(x_bar, color='#e74c3c', lw=2, linestyle='--',
            label=f'x̄ = {x_bar:.0f}')
ax1.set_title('Traffic Flow Differences (6th − 13th)')
ax1.set_xlabel('Difference (vehicle count)')
ax1.set_ylabel('Frequency')
ax1.legend()

# Right: t-distribution with T statistic and p-value
ax2 = axes[1]
x_range = np.linspace(-6, 6, 500)
y_t     = t.pdf(x_range, df=df)
ax2.plot(x_range, y_t, 'k-', lw=2)

# Shade p-value regions
tail_r = x_range[x_range >= T_obs]
tail_l = x_range[x_range <= -T_obs]
ax2.fill_between(tail_r, t.pdf(tail_r, df=df), alpha=0.4,
                 color='#e74c3c', label=f'p-value = {p_val:.4f}')
ax2.fill_between(tail_l, t.pdf(tail_l, df=df), alpha=0.4, color='#e74c3c')
ax2.axvline( T_obs, color='#e74c3c', lw=2, linestyle='--',
             label=f'T = {T_obs:.2f}')
ax2.axvline(-T_obs, color='#e74c3c', lw=2, linestyle='--')
ax2.set_title(f't-Distribution (df={df}) — Hypothesis Test')
ax2.set_xlabel('t value')
ax2.set_ylabel('Density')
ax2.legend()

plt.tight_layout()
plt.suptitle('Figure 2: Friday the 13th — One-Sample t-Test',
             fontsize=13, fontweight='bold', y=1.02)
plt.show()

In [ ]:
# 95% Confidence Interval
alpha   = 0.05
t_star  = t.ppf(1 - alpha/2, df=df)   # critical t value
margin  = t_star * se
ci_low  = x_bar - margin
ci_high = x_bar + margin

print('=== 95% Confidence Interval ===')
print(f'Critical t* (df=9, α/2=0.025): {t_star:.4f}')
print(f'Margin of error (ME = t* × SE): {margin:.2f}')
print(f'CI: {x_bar:.0f} ± {margin:.0f}')
print(f'CI: ({ci_low:.0f}, {ci_high:.0f})')
print()
print('Interpretation: We are 95% confident that the true mean')
print(f'traffic difference between the 6th and 13th Friday lies between {ci_low:.0f} and {ci_high:.0f} vehicles.')
print()
print(f'Does the CI contain zero? {"Yes" if ci_low <= 0 <= ci_high else "No"}')
print('→ The HT and CI results are consistent!')

---
## 3. Paired Data (Paired t-test): Reading & Writing Scores

### Why Paired?

The same students took both the reading and writing tests. Therefore the two measurements are **dependent** — there is one pair of observations per student.

**Strategy:** Compute differences → apply a one-sample t-test

$$\text{diff}_i = \text{reading}_i - \text{writing}_i$$

$$H_0: \mu_{diff} = 0, \quad H_A: \mu_{diff} \neq 0$$

In [ ]:
# Simulate realistic data (consistent with the statistics in the slides)
# x̄_diff = -0.545, s_diff = 8.887, n = 200
np.random.seed(42)
n_students = 200

# Multivariate normal with mean vector and covariance matrix
mu  = [51.0, 51.5]
cov = [[100, 75],   # reading and writing are positively correlated
       [75,  90]]
scores = np.random.multivariate_normal(mu, cov, n_students)
scores = np.clip(np.round(scores), 20, 80).astype(int)

df_scores = pd.DataFrame({'reading': scores[:,0], 'writing': scores[:,1]})
df_scores['diff'] = df_scores['reading'] - df_scores['writing']

print('First 5 rows:')
print(df_scores.head())
print(f'\nReading → Mean: {df_scores["reading"].mean():.2f}, SD: {df_scores["reading"].std():.2f}')
print(f'Writing → Mean: {df_scores["writing"].mean():.2f}, SD: {df_scores["writing"].std():.2f}')
print(f'Diff    → Mean: {df_scores["diff"].mean():.3f}, SD: {df_scores["diff"].std():.3f}')

In [ ]:
# Repeat the analysis using the values from the slides
x_bar_diff = -0.545
s_diff     =  8.887
n_students =  200
se_diff    = s_diff / np.sqrt(n_students)
df_paired  = n_students - 1
T_paired   = x_bar_diff / se_diff
p_paired   = 2 * t.sf(abs(T_paired), df=df_paired)

print('=== Paired t-Test: Reading − Writing ===')
print(f'Mean difference:       {x_bar_diff}')
print(f'Std deviation (diff):  {s_diff}')
print(f'n:                     {n_students}')
print(f'Standard error (SE):   {se_diff:.4f}')
print(f'T statistic:           {T_paired:.4f}')
print(f'Degrees of freedom:    {df_paired}')
print(f'p-value:               {p_paired:.4f}')
print(f'Decision (α=0.05):     {"REJECT H₀" if p_paired < 0.05 else "FAIL TO REJECT H₀ ✓"}')
print()
print('Interpretation: The data do NOT provide convincing evidence of a')
print('significant difference between mean reading and writing scores.')

# 95% CI
t_star_p = t.ppf(0.975, df=df_paired)
ci_p     = (x_bar_diff - t_star_p*se_diff, x_bar_diff + t_star_p*se_diff)
print(f'\n95% CI: ({ci_p[0]:.3f}, {ci_p[1]:.3f})')
print('→ CI contains zero → consistent with HT.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Left: Box plots
ax1 = axes[0]
ax1.boxplot([df_scores['reading'], df_scores['writing']],
            labels=['Reading', 'Writing'],
            patch_artist=True,
            boxprops=dict(facecolor='#3498db', alpha=0.6),
            medianprops=dict(color='black', lw=2))
ax1.set_title('Reading and Writing Scores')
ax1.set_ylabel('Score')

# Middle: Histogram of differences
ax2 = axes[1]
ax2.hist(df_scores['diff'], bins=20, color='#2ecc71', alpha=0.7,
         edgecolor='white')
ax2.axvline(df_scores['diff'].mean(), color='#e74c3c', lw=2,
            linestyle='--', label=f'x̄ = {df_scores["diff"].mean():.2f}')
ax2.axvline(0, color='black', lw=1.5, linestyle=':')
ax2.set_title('Distribution of Differences (Reading − Writing)')
ax2.set_xlabel('Difference')
ax2.set_ylabel('Frequency')
ax2.legend()

# Right: t-distribution and p-value
ax3 = axes[2]
x_r = np.linspace(-4, 4, 500)
ax3.plot(x_r, t.pdf(x_r, df=df_paired), 'k-', lw=2)
ax3.axvline( T_paired, color='#e74c3c', lw=2, linestyle='--',
             label=f'T = {T_paired:.2f}')
ax3.axvline(-T_paired, color='#e74c3c', lw=2, linestyle='--')
# Tails
t_lo = x_r[x_r <= T_paired]
t_hi = x_r[x_r >= -T_paired]
ax3.fill_between(t_lo, t.pdf(t_lo, df=df_paired), alpha=0.4, color='#e74c3c')
ax3.fill_between(t_hi, t.pdf(t_hi, df=df_paired), alpha=0.4, color='#e74c3c',
                 label=f'p = {p_paired:.4f}')
ax3.set_title('Hypothesis Test')
ax3.set_xlabel('t value')
ax3.set_ylabel('Density')
ax3.legend()

plt.tight_layout()
plt.suptitle('Figure 3: Paired t-Test — Reading & Writing',
             fontsize=13, fontweight='bold', y=1.02)
plt.show()

---
## 4. Two Independent Samples t-Test: Diamond Prices

### Research Question
Despite an imperceptible size difference between 0.99 and 1 carat diamonds, **is there a price difference?**

### Hypotheses (One-sided)
$$H_0: \mu_{pt99} = \mu_{pt100}$$
$$H_A: \mu_{pt99} < \mu_{pt100} \quad (\text{0.99 carat is cheaper})$$

### Test Statistic
$$T_{df} = \frac{(\bar{x}_1 - \bar{x}_2) - 0}{\sqrt{\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}}}, \quad df = \min(n_1-1, n_2-1)$$

In [ ]:
# Summary statistics from the slides
x_bar_99  = 44.50;  s_99  = 13.32;  n_99  = 23
x_bar_100 = 53.43;  s_100 = 12.22;  n_100 = 30

# Simulate realistic data (consistent with summary statistics)
np.random.seed(7)
pt99  = np.random.normal(x_bar_99,  s_99,  n_99 ).round(1)
pt100 = np.random.normal(x_bar_100, s_100, n_100).round(1)

print('0.99 carat:', pt99)
print('\n1 carat:  ', pt100)

# Standard error and T statistic
se_two = np.sqrt(s_99**2/n_99 + s_100**2/n_100)
df_two = min(n_99 - 1, n_100 - 1)    # = 22
T_two  = (x_bar_99 - x_bar_100) / se_two
p_two  = t.cdf(T_two, df=df_two)      # one-sided (left tail)

print('\n=== Two Independent Samples t-Test: Diamond Prices ===')
print(f'         0.99 carat  |  1 carat')
print(f'Mean:    {x_bar_99:.2f}      |  {x_bar_100:.2f}')
print(f'SD:      {s_99:.2f}      |  {s_100:.2f}')
print(f'n:       {n_99}          |  {n_100}')
print(f'\nSE:                  {se_two:.4f}')
print(f'T statistic:         {T_two:.4f}')
print(f'Degrees of freedom:  {df_two}')
print(f'p-value (one-sided): {p_two:.6f}')
print(f'\nDecision (α=0.05):   {"REJECT H₀ ✓" if p_two < 0.05 else "FAIL TO REJECT H₀"}')

In [ ]:
# 90% Confidence Interval (equivalent to one-sided α=0.05)
t_star_90 = t.ppf(0.95, df=df_two)   # = 1.717
diff_hat  = x_bar_99 - x_bar_100
ci_90     = (diff_hat - t_star_90*se_two, diff_hat + t_star_90*se_two)

print('=== 90% Confidence Interval (equivalent to one-sided α=0.05) ===')
print(f'Point estimate:      {diff_hat:.2f}')
print(f'Critical t* (df=22): {t_star_90:.4f}')
print(f'90% CI:              ({ci_90[0]:.2f}, {ci_90[1]:.2f})')
print()
print('Interpretation: We are 90% confident that the average price')
print(f'per point for a 0.99-carat diamond is between ${abs(ci_90[0]):.2f} and ${abs(ci_90[1]):.2f}')
print('lower than that of a 1-carat diamond.')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax1 = axes[0]
data_box   = [pt99, pt100]
bp = ax1.boxplot(data_box, labels=['0.99 carat', '1 carat'],
                 patch_artist=True, notch=False,
                 boxprops=dict(alpha=0.7),
                 medianprops=dict(color='black', lw=2))
colors_box = ['#3498db', '#e74c3c']
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
ax1.scatter([1]*n_99 + [2]*n_100,
            np.concatenate([pt99, pt100]),
            alpha=0.4, color=['#3498db']*n_99 + ['#e74c3c']*n_100,
            zorder=5, s=25)
ax1.set_title('Diamond Prices per Point')
ax1.set_ylabel('Price ($/point)')

ax2 = axes[1]
x_r = np.linspace(-4.5, 1.5, 500)
ax2.plot(x_r, t.pdf(x_r, df=df_two), 'k-', lw=2, label=f't(df={df_two})')
tail = x_r[x_r <= T_two]
ax2.fill_between(tail, t.pdf(tail, df=df_two), alpha=0.4, color='#e74c3c',
                 label=f'p = {p_two:.4f}')
ax2.axvline(T_two, color='#e74c3c', lw=2, linestyle='--',
            label=f'T = {T_two:.2f}')
ax2.set_title('Hypothesis Test (One-Sided)')
ax2.set_xlabel('t value')
ax2.set_ylabel('Density')
ax2.legend()

plt.tight_layout()
plt.suptitle('Figure 4: Two Independent Samples t-Test — Diamond Prices',
             fontsize=13, fontweight='bold', y=1.02)
plt.show()

---
## 5. Power Analysis (Two-Sample Test)

### Decision Table

| | Fail to Reject H₀ | Reject H₀ |
|---|---|---|
| **H₀ True** | ✓ (1−α) | Type I Error (α) |
| **Hₐ True** | Type II Error (β) | ✓ **Power** (1−β) |

### Example: Blood Pressure Clinical Trial

- σ = 12 mmHg, n = 100 per group, target effect δ = −3 mmHg
- SE = √(12²/100 + 12²/100) = 1.70

In [ ]:
sigma  = 12
n_each = 100
alpha  = 0.05
delta  = -3   # true effect size

se_bp       = np.sqrt(sigma**2/n_each + sigma**2/n_each)
z_crit      = norm.ppf(1 - alpha/2)   # = 1.96

# Rejection regions (two-sided)
reject_low  = -z_crit * se_bp    # ≈ -3.332
reject_high =  z_crit * se_bp    # ≈ +3.332

# Power: real distribution is centred at delta while H₀ is rejected
z_power = (reject_low - delta) / se_bp
power   = norm.cdf(z_power)

print('=== Blood Pressure Clinical Trial — Power Analysis ===')
print(f'σ = {sigma}, n = {n_each}/group, δ = {delta} mmHg, α = {alpha}')
print(f'Standard error (SE):   {se_bp:.4f}')
print(f'Critical value (z*):   ±{z_crit:.4f}')
print(f'Rejection region:      < {reject_low:.3f} or > {reject_high:.3f}')
print(f'Z (power calc):        {z_power:.4f}')
print(f'Power (1-β):           {power:.4f} = {power*100:.1f}%')
print(f'Type II error (β):     {1-power:.4f} = {(1-power)*100:.1f}%')

In [ ]:
# Power curve: power for different sample sizes
n_values  = np.array([10, 20, 30, 50, 75, 100, 150, 200, 300, 500, 750, 1000])
powers_bp = []

for n_val in n_values:
    se_val  = np.sqrt(sigma**2/n_val + sigma**2/n_val)
    rej_low = -z_crit * se_val
    z_p     = (rej_low - delta) / se_val
    powers_bp.append(norm.cdf(z_p))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Power curve
ax1 = axes[0]
ax1.semilogx(n_values, powers_bp, 'b-o', lw=2, markersize=6)
ax1.axhline(0.80, color='#e74c3c', lw=1.5, linestyle='--', label='80% power target')
ax1.axhline(0.90, color='#2ecc71', lw=1.5, linestyle='--', label='90% power target')
ax1.set_xlabel('Sample size per group (log scale)')
ax1.set_ylabel('Power (1−β)')
ax1.set_title('Power Curve — Blood Pressure Trial')
ax1.set_ylim(0, 1.05)
ax1.legend()
ax1.grid(True, alpha=0.4)

# Right: Null and alternative distributions
ax2 = axes[1]
x_plot   = np.linspace(-10, 4, 600)
null_dist = norm.pdf(x_plot, 0, se_bp)
alt_dist  = norm.pdf(x_plot, delta, se_bp)
ax2.plot(x_plot, null_dist, 'b-', lw=2, label='H₀ distribution (δ=0)')
ax2.plot(x_plot, alt_dist,  'g-', lw=2, label=f'Hₐ distribution (δ={delta})')
# Power region
power_region = x_plot[x_plot <= reject_low]
ax2.fill_between(power_region, norm.pdf(power_region, delta, se_bp),
                 alpha=0.4, color='green', label=f'Power = {power:.2f}')
# Type I region
alpha_region = x_plot[x_plot <= reject_low]
ax2.fill_between(alpha_region, norm.pdf(alpha_region, 0, se_bp),
                 alpha=0.3, color='blue', label=f'α/2 = {alpha/2}')
ax2.axvline(reject_low,  color='red', lw=1.5, linestyle='--', label=f'Rejection boundary={reject_low:.2f}')
ax2.axvline(reject_high, color='red', lw=1.5, linestyle='--')
ax2.set_xlabel('x̄_treatment − x̄_control')
ax2.set_ylabel('Density')
ax2.set_title('Null and Alternative Distributions')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.suptitle('Figure 5: Power Analysis', fontsize=13, fontweight='bold', y=1.02)
plt.show()

In [ ]:
# Required sample size for 80% power
# SE = δ / (z_α/2 + z_β)  →  n = 2σ²(z_α/2 + z_β)² / δ²
target_power = 0.80
z_beta  = norm.ppf(target_power)     # = 0.842
n_req   = 2 * sigma**2 * (z_crit + z_beta)**2 / delta**2

print('=== Required Sample Size for 80% Power ===')
print(f'z_α/2 = {z_crit:.3f}')
print(f'z_β   = {z_beta:.3f}')
print(f'δ = {delta}, σ = {sigma}')
print(f'n calculation: 2 × {sigma}² × ({z_crit:.3f} + {z_beta:.3f})² / {delta}²')
print(f'n = {n_req:.2f}  →  n ≥ {int(np.ceil(n_req))}')
print('(Consistent with slides result: n ≥ 251)')

---
## 6. ANOVA: Wolf River Aldrin Concentrations

### Research Question
Do mean aldrin concentrations differ across three depth levels (bottom, mid-depth, surface)?

### Hypotheses
$$H_0: \mu_B = \mu_M = \mu_S$$
$$H_A: \text{At least one mean differs}$$

### Logic of ANOVA

$$F = \frac{\text{Between-group variability (MSG)}}{\text{Within-group variability (MSE)}}$$

Large F → large between-group difference → reject H₀

In [ ]:
# Wolf River Aldrin Data — consistent with slide values
np.random.seed(2024)

# Group statistics: n=10, means: 6.04, 5.05, 4.20
bottom   = np.array([3.80, 4.80, 4.90, 5.60, 6.50, 6.00, 7.80, 8.80, 6.50, 5.30])
middepth = np.array([3.20, 3.80, 4.00, 5.30, 5.50, 6.40, 6.60, 4.90, 5.80, 5.30])
surface  = np.array([3.10, 3.60, 3.80, 4.00, 4.20, 4.60, 4.90, 5.20, 4.30, 3.80])

df_aldrin = pd.DataFrame({
    'aldrin': np.concatenate([bottom, middepth, surface]),
    'depth':  ['bottom']*10 + ['mid-depth']*10 + ['surface']*10
})

print('=== Summary Statistics ===')
summary = df_aldrin.groupby('depth')['aldrin'].agg(['count','mean','std'])
summary.columns = ['n', 'Mean', 'Std Dev']
summary.loc['Overall'] = [30, df_aldrin['aldrin'].mean(), df_aldrin['aldrin'].std()]
print(summary.round(2))

In [ ]:
# ANOVA — Manual Calculation
groups      = [bottom, middepth, surface]
k           = len(groups)
n_total     = sum(len(g) for g in groups)
grand_mean  = np.concatenate(groups).mean()
group_means = [g.mean() for g in groups]
group_ns    = [len(g)   for g in groups]

# Degrees of freedom
df_G = k - 1
df_T = n_total - 1
df_E = df_T - df_G

# Sums of squares
SSG = sum(n_i * (x_i - grand_mean)**2
          for n_i, x_i in zip(group_ns, group_means))
SST = sum((x - grand_mean)**2 for x in np.concatenate(groups))
SSE = SST - SSG

# Mean squares
MSG = SSG / df_G
MSE = SSE / df_E

# F statistic and p-value
F_obs   = MSG / MSE
p_anova = 1 - stats.f.cdf(F_obs, dfn=df_G, dfd=df_E)

print('=== ANOVA Table (Manual Calculation) ===')
print(f'{"":<15} {"df":>5} {"Sum of Sq":>12} {"Mean Sq":>10} {"F value":>10} {"Pr(>F)":>10}')
print('-'*70)
print(f'{"(Group) depth":<15} {df_G:>5} {SSG:>12.2f} {MSG:>10.2f} {F_obs:>10.2f} {p_anova:>10.4f}')
print(f'{"(Error) Residuals":<15} {df_E:>5} {SSE:>12.2f} {MSE:>10.2f}')
print(f'{"Total":<15} {df_T:>5} {SST:>12.2f}')
print()
print(f'Decision (α=0.05): {"REJECT H₀ ✓" if p_anova < 0.05 else "FAIL TO REJECT H₀"}')
print('Interpretation: The mean aldrin concentration at at least one depth level')
print('differs significantly from the others.')

In [ ]:
# Verification with scipy
F_scipy, p_scipy = stats.f_oneway(bottom, middepth, surface)
print(f'scipy.stats.f_oneway → F = {F_scipy:.4f}, p = {p_scipy:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Left: Dot plot (matching the slide visual)
ax1 = axes[0]
colors_d = {'bottom': '#e74c3c', 'mid-depth': '#3498db', 'surface': '#2ecc71'}
y_pos    = {'bottom': 2, 'mid-depth': 1, 'surface': 0}
y_labels = ['Surface', 'Mid-Depth', 'Bottom']

for group_name, group_data in zip(['bottom','mid-depth','surface'], groups):
    yp = y_pos[group_name]
    ax1.scatter(group_data, [yp]*len(group_data),
                color=colors_d[group_name], alpha=0.7, s=60, zorder=5)
    ax1.hlines(yp, group_data.min(), group_data.max(),
               color=colors_d[group_name], lw=1.5, alpha=0.5)
    m = group_data.mean()
    ax1.scatter(m, yp, color=colors_d[group_name], s=120,
                marker='D', zorder=6, edgecolor='white', lw=1)

ax1.set_yticks([0, 1, 2])
ax1.set_yticklabels(y_labels)
ax1.set_xlabel('Aldrin Concentration (ng/L)')
ax1.set_title('Three Depth Levels')

# Middle: Box plots + variance check
ax2 = axes[1]
bp = ax2.boxplot(groups, labels=['Bottom\nsd=1.58', 'Mid-Depth\nsd=1.10', 'Surface\nsd=0.66'],
                 patch_artist=True,
                 medianprops=dict(color='black', lw=2))
for patch, color in zip(bp['boxes'], ['#e74c3c', '#3498db', '#2ecc71']):
    patch.set_facecolor(color); patch.set_alpha(0.6)
ax2.set_ylabel('Aldrin (ng/L)')
ax2.set_title('Box Plots (Constant Variance Check)')

# Right: F-distribution and p-value
ax3 = axes[2]
x_f = np.linspace(0.001, 10, 500)
ax3.plot(x_f, stats.f.pdf(x_f, df_G, df_E), 'k-', lw=2,
         label=f'F(df₁={df_G}, df₂={df_E})')
tail_f = x_f[x_f >= F_obs]
ax3.fill_between(tail_f, stats.f.pdf(tail_f, df_G, df_E),
                 alpha=0.4, color='#e74c3c', label=f'p = {p_anova:.4f}')
ax3.axvline(F_obs, color='#e74c3c', lw=2, linestyle='--',
            label=f'F = {F_obs:.2f}')
ax3.set_xlabel('F value')
ax3.set_ylabel('Density')
ax3.set_title('F-Distribution — p-Value')
ax3.set_xlim(0, 10)
ax3.legend()

plt.tight_layout()
plt.suptitle('Figure 6: ANOVA — Wolf River Aldrin Concentrations',
             fontsize=13, fontweight='bold', y=1.02)
plt.show()

---
## 7. Multiple Comparisons — Bonferroni Correction

After rejecting H₀ in ANOVA, the question **which groups** differ arises. Running a t-test for every pair inflates the Type I error rate.

### Bonferroni Correction

$$\alpha^* = \frac{\alpha}{K}, \quad K = \frac{k(k-1)}{2}$$

For 3 groups, K = 3 comparisons → α* = 0.05/3 = 0.0167

### Pooled Standard Error

$$SE = \sqrt{\frac{MSE}{n_1} + \frac{MSE}{n_2}}, \quad df = n - k$$

In [ ]:
alpha_bonf = 0.05
K          = k * (k - 1) // 2
alpha_star = alpha_bonf / K

print('=== Multiple Comparisons (Bonferroni) ===')
print(f'Number of groups:      k = {k}')
print(f'Number of comparisons: K = k(k-1)/2 = {K}')
print(f'Corrected α* = {alpha_bonf}/{K} = {alpha_star:.4f}')
print()

# t-test for all pairs (using pooled SE)
group_names = ['bottom', 'mid-depth', 'surface']
pairs       = [('bottom','mid-depth'), ('bottom','surface'), ('mid-depth','surface')]
group_dict  = {'bottom': bottom, 'mid-depth': middepth, 'surface': surface}
df_error    = n_total - k   # = 27

print(f'{"Comparison":<25} {"T":>8} {"p (two-sided)":>16} {"α*=0.0167":>12} {"Decision":>20}')
print('-'*85)

results = []
for g1, g2 in pairs:
    d1, d2   = group_dict[g1], group_dict[g2]
    se_pool  = np.sqrt(MSE/len(d1) + MSE/len(d2))
    T_ij     = (d1.mean() - d2.mean()) / se_pool
    p_ij     = 2 * t.sf(abs(T_ij), df=df_error)
    decision = 'Reject H₀ ✓' if p_ij < alpha_star else 'Fail to Reject H₀'
    results.append({'pair': f'{g1} vs {g2}', 'T': T_ij, 'p': p_ij, 'decision': decision})
    print(f'{g1:>10} vs {g2:<12} {T_ij:>8.3f} {p_ij:>16.4f} {"< 0.0167" if p_ij < alpha_star else "> 0.0167":>12} {decision:>20}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Comparison p-values vs Bonferroni threshold
ax1         = axes[0]
pair_labels = [r['pair'] for r in results]
p_vals_all  = [r['p']    for r in results]
bar_colors  = ['#e74c3c' if p < alpha_star else '#95a5a6' for p in p_vals_all]
bars = ax1.bar(pair_labels, p_vals_all, color=bar_colors, alpha=0.8, edgecolor='white')
ax1.axhline(alpha_star, color='navy', lw=2, linestyle='--',
            label=f'α* = {alpha_star:.4f}')
ax1.axhline(0.05, color='gray', lw=1.5, linestyle=':',
            label='α = 0.05')
ax1.set_ylabel('p-value')
ax1.set_title('Bonferroni-Corrected Comparisons')
ax1.legend()
for bar, pv in zip(bars, p_vals_all):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             f'{pv:.4f}', ha='center', va='bottom', fontsize=9)

# Right: Mean differences and confidence intervals
ax2          = axes[1]
t_star_bonf  = t.ppf(1 - alpha_star/2, df=df_error)
diffs        = []
ci_lo_list   = []
ci_hi_list   = []

for g1, g2 in pairs:
    d1, d2  = group_dict[g1], group_dict[g2]
    diff    = d1.mean() - d2.mean()
    se_pool = np.sqrt(MSE/len(d1) + MSE/len(d2))
    margin  = t_star_bonf * se_pool
    diffs.append(diff)
    ci_lo_list.append(diff - margin)
    ci_hi_list.append(diff + margin)

y_pos_c = range(len(pairs))
for i, (diff, lo, hi, res) in enumerate(
        zip(diffs, ci_lo_list, ci_hi_list, results)):
    color = '#e74c3c' if 'Reject' in res['decision'] and '✓' in res['decision'] \
            else '#95a5a6'
    ax2.plot([lo, hi], [i, i], color=color, lw=3, alpha=0.8)
    ax2.scatter(diff, i, color=color, s=100, zorder=5)

ax2.axvline(0, color='black', lw=1.5, linestyle='--')
ax2.set_yticks(list(y_pos_c))
ax2.set_yticklabels([r['pair'] for r in results])
ax2.set_xlabel('Mean Difference')
ax2.set_title(f'Confidence Intervals (Bonferroni, α*={alpha_star:.4f})')
red_patch = mpatches.Patch(color='#e74c3c', label='Reject H₀')
gray_patch= mpatches.Patch(color='#95a5a6', label='Fail to Reject H₀')
ax2.legend(handles=[red_patch, gray_patch])

plt.tight_layout()
plt.suptitle('Figure 7: Multiple Comparisons — Bonferroni Correction',
             fontsize=13, fontweight='bold', y=1.02)
plt.show()

print('\nConclusion: Only the bottom & surface difference is statistically significant.')
print('The bottom & mid-depth and mid-depth & surface differences remain non-significant after the Bonferroni correction.')

---
## Summary: Which Test Should I Use and When?

| Situation | Test | Statistic |
|---|---|---|
| 1 group, σ unknown | One-sample t-test | $T = \frac{\bar{x} - \mu_0}{s/\sqrt{n}}$ |
| 2 dependent groups (paired) | Paired t-test | One-sample t on differences |
| 2 independent groups, σ unknown | Two-sample t-test | $T = \frac{(\bar{x}_1-\bar{x}_2)}{\sqrt{s_1^2/n_1+s_2^2/n_2}}$ |
| ≥3 independent groups | ANOVA (F-test) | $F = MSG/MSE$ |
| Post-ANOVA pairwise | t-test + Bonferroni | α* = α/K |

In [ ]:
print('Notebook complete!')
print('Topics covered:')
topics = [
    '1. The t-Distribution — Differences from Normal',
    '2. One-Sample t-Test (Friday the 13th)',
    '3. Confidence Interval Calculation',
    '4. Paired t-Test (Reading & Writing)',
    '5. Two Independent Samples t-Test (Diamond Prices)',
    '6. Power Analysis & Sample Size',
    '7. ANOVA (Wolf River Aldrin)',
    '8. Multiple Comparisons (Bonferroni)'
]
for topic in topics:
    print(f'  ✓ {topic}')